# 🚀 CinemaBox 1-Click AI Cloud Server (5 TB Google Storage)
### Powered by Qwen 2.5 32B Instruct (128k Context | Direct 5TB Drive Streaming)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/princesharmaofficial1-hub/cinemabox-colab/blob/main/deploy_70b_google_storage.ipynb)

---

### 🧹 STEP 0: Deep Clean Colab Disk (Instant Free Space)
Click the **Play** button below to wipe all temporary cache and junk from the small Colab disk.

In [ ]:
#@title 🧹 Click to Wipe Cache & Free Colab Disk { vertical-output: true }
print("🧹 Cleaning Colab root disk...")
!rm -rf /root/.cache/* /tmp/* /content/*.deb /content/sample_data /root/.local/share/Trash/*
print("\n✅ Local disk cleaned! Available space:")
!df -h /

#@title 🚀 1-Click Launch Heavy AI Server (5 TB Google Drive + Cloudflare) { vertical-output: true }
#@markdown ### 🧠 Select Your Heavy Model
Model_Choice = "Qwen 2.5 32B Instruct (Fast & SOTA 128k)" #@param ["Qwen 2.5 32B Instruct (Fast & SOTA 128k)", "Qwen 2.5 72B (World #1 Open Dense Reasoning)", "Mixtral 8x22B (141B Heavyweight MoE)", "LLaMA 3.3 70B Instruct", "Qwen 2 VL 72B (Multi-Modal Vision)"]
Context_Window = 4096 #@param [2048, 4096, 8192, 16384, 32768]
GPU_Layers = 28 #@param [15, 20, 25, 28, 30, 35, -1]
API_Key = "sk-cinemabox70b" #@param {type:"string"}

import os
import sys
import time
import subprocess
import re
import urllib.request
import json

print("="*75)
print("🚀 CINEMABOX CLOUD AI ENGINE (5 TB GOOGLE DRIVE STORAGE)")
print(f"   Model:   {Model_Choice}")
print(f"   Context: {Context_Window} tokens | GPU Offload: {GPU_Layers} layers")
print("="*75)

# 1. Mount 5 TB Google Drive
print("\n[1/4] 📁 Mounting your 5 TB Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/CinemaBox_Models'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"      ✅ Connected to 5 TB Google Drive: {DRIVE_DIR}")

# 2. Install High-Speed GPU Engine & Cloudflare Tunnel
print("\n[2/4] ⚡ Installing GPU Engine & Cloudflare Tunnel...")
!pip install --upgrade pip -q
!pip install -q "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!pip install -q pydantic-settings sse-starlette starlette-context fastapi uvicorn requests jinja2

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("      ✅ GPU Engine & Cloudflare ready!")

# 3. Model Catalog & Direct Google Drive Streaming
MODEL_CATALOG = {
    "Mixtral 8x22B (141B Heavyweight MoE)": {
        "file": "Mixtral-8x22B-Instruct-v0.1-Q4_K_M.gguf",
        "url": "https://huggingface.co/bartowski/Mixtral-8x22B-Instruct-v0.1-GGUF/resolve/main/Mixtral-8x22B-Instruct-v0.1-Q4_K_M.gguf",
        "id": "Mixtral-8x22B-Instruct"
    },
    "Qwen 2.5 72B (World #1 Open Dense Reasoning)": {
        "file": "Qwen2.5-72B-Instruct-Q4_K_M.gguf",
        "url": "https://huggingface.co/bartowski/Qwen2.5-72B-Instruct-GGUF/resolve/main/Qwen2.5-72B-Instruct-Q4_K_M.gguf",
        "id": "Qwen2.5-72B-Instruct"
    },
    "LLaMA 3.3 70B Instruct": {
        "file": "Llama-3.3-70B-Instruct-Q4_K_M.gguf",
        "url": "https://huggingface.co/bartowski/Llama-3.3-70B-Instruct-GGUF/resolve/main/Llama-3.3-70B-Instruct-Q4_K_M.gguf",
        "id": "Llama-3.3-70B-Instruct"
    },
    "Qwen 2 VL 72B (Multi-Modal Vision)": {
        "file": "Qwen2-VL-72B-Instruct-Q4_K_M.gguf",
        "url": "https://huggingface.co/bartowski/Qwen2-VL-72B-Instruct-GGUF/resolve/main/Qwen2-VL-72B-Instruct-Q4_K_M.gguf",
        "id": "Qwen2-VL-72B-Instruct"
    },
    "Qwen 2.5 32B Instruct (Fast & SOTA 128k)": {
        "file": "Qwen2.5-32B-Instruct-Q4_K_M.gguf",
        "url": "https://huggingface.co/bartowski/Qwen2.5-32B-Instruct-GGUF/resolve/main/Qwen2.5-32B-Instruct-Q4_K_M.gguf",
        "id": "Qwen2.5-32B-Instruct"
    }
}

target = MODEL_CATALOG[Model_Choice]
model_path = os.path.join(DRIVE_DIR, target["file"])

print(f"\n[3/4] 📦 Resolving Model in 5 TB Google Drive: {target['file']}...")
if os.path.exists(model_path) and os.path.getsize(model_path) > 1024*1024*100:
    print(f"      🎯 Reusing permanently cached model in 5 TB Drive: {target['file']}")
    print(f"         Size: {os.path.getsize(model_path)/(1024**3):.2f} GB")
else:
    print(f"      📥 Streaming {target['file']} directly into 5 TB Google Drive...")
    print("         (Streaming at 100+ MB/s directly to Drive with zero local disk cache)")
    !wget -c --show-progress -O "{model_path}" "{target['url']}"
    print(f"      ✅ Saved permanently to: {model_path}")

# 4. Launch OpenAI Server & Wait for GPU Weights Initialization
print("\n[4/4] 🌐 Launching OpenAI Server & Warming CUDA GPU Weights...")

# Ensure any previous instances are cleared
!pkill -f llama_cpp.server > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

server_log = open('/content/llama_server.log', 'w')
server_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", model_path,
    "--n_gpu_layers", str(GPU_Layers),
    "--n_ctx", str(Context_Window),
    "--host", "0.0.0.0",
    "--port", "8000",
    "--api_key", API_Key
]

server_proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)

print("      ⏳ Loading weights into GPU memory (takes ~30-50s)...")
server_ready = False
for attempt in range(60):
    time.sleep(2)
    if server_proc.poll() is not None:
        print("\n❌ Server process exited! Reading error log:")
        with open('/content/llama_server.log', 'r') as f:
            print(f.read())
        break
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/v1/models")
        req.add_header("Authorization", f"Bearer {API_Key}")
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                print("      ✅ GPU Server is 100% loaded and responding on port 8000!")
                break
    except Exception:
        if attempt % 5 == 0 and attempt > 0:
            print(f"      ... warming up CUDA layers ({attempt*2}s elapsed)")

if not server_ready:
    print("\n⚠️ Server did not respond in time. Checking log output:")
    with open('/content/llama_server.log', 'r') as f:
        print(f.read()[-1500:])

# Start Cloudflare Tunnel with persistent file logging (prevents SIGPIPE crash)
print("\n      🌐 Establishing Cloudflare Global Tunnel...")
cf_log = open('/content/cloudflared.log', 'w')
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=cf_log,
    stderr=subprocess.STDOUT
)

public_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists('/content/cloudflared.log'):
        with open('/content/cloudflared.log', 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("\n" + "="*75)
    print("🎉 YOUR HEAVY AI SERVER IS LIVE GLOBALLY AND READY TO USE!")
    print("="*75)
    print(f"\n👉 📱 CinemaBox App Endpoint:     {public_url}/v1/chat/completions")
    print(f"👉 💻 VS Code (Roo/Continue) Base: {public_url}/v1")
    print(f"👉 🔑 API Key:                     {API_Key}")
    print(f"👉 🧠 Active Model:                {target['id']}")
    print("\n" + "-"*75)
    print("📱 HOW TO USE IN CINEMABOX APP:")
    print("   1. Open CinemaBox on phone > Settings > On-Device Intelligence")
    print("   2. Tap '141B MoE / VISION' tab")
    print(f"   3. Paste '{public_url}/v1/chat/completions' into Host API Endpoint")
    print("   4. Tap Save & Run Test -> You are connected!")
    print("\n💻 HOW TO USE IN VS CODE (Roo Code / Continue / Cline):")
    print("   Provider: OpenAI Compatible")
    print(f"   Base URL: {public_url}/v1")
    print(f"   API Key:  {API_Key}")
    print(f"   Model:    {target['id']}")
    print("="*75 + "\n")
    print("🟢 SERVER RUNNING ACTIVELY (Keep this Colab tab open while using the app)...")
    try:
        while True:
            if server_proc.poll() is not None:
                print("\n❌ Server stopped! Log:")
                with open('/content/llama_server.log', 'r') as f:
                    print(f.read()[-1500:])
                break
            if cf_proc.poll() is not None:
                print("\n❌ Cloudflare tunnel stopped! Log:")
                with open('/content/cloudflared.log', 'r') as f:
                    print(f.read()[-1500:])
                break
            time.sleep(5)
    except KeyboardInterrupt:
        print("\n🛑 Shutting down server...")
        server_proc.terminate()
        cf_proc.terminate()
else:
    print("❌ Cloudflare tunnel could not be established. Check /content/cloudflared.log")



In [ ]:
#@title 🚀 1-Click Launch AI Server & Cloudflare Tunnel { vertical-output: true }
import os
import sys
import time
import subprocess
import re

print("="*70)
print("🚀 STARTING CINEMABOX CLOUD AI SERVER...")
print("="*70)

# 1. Mount 5 TB Google Drive
print("\n[1/4] 📁 Mounting your 5 TB Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/CinemaBox_Models'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"      ✅ Connected to 5 TB Google Drive: {DRIVE_DIR}")

# 2. Install High-Speed GPU Engine, aria2 & Cloudflare Tunnel
print("\n[2/4] ⚡ Installing GPU Engine, aria2 & Cloudflare Tunnel...")
!apt-get install -y aria2 -qq > /dev/null 2>&1
!pip install --upgrade pip -q
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q
!pip install "llama-cpp-python[server]" fastapi uvicorn requests -q

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("      ✅ GPU Engine & Cloudflare ready!")

# 3. Stream Model Directly into 5 TB Google Drive (Zero local disk usage)
print("\n[3/4] 📦 Storing Qwen 2.5 32B in 5 TB Google Drive (128k Context, SOTA Intelligence)...")
MODEL_FILENAME = "Qwen2.5-32B-Instruct-Q4_K_M.gguf"
MODEL_URL = "https://huggingface.co/bartowski/Qwen2.5-32B-Instruct-GGUF/resolve/main/Qwen2.5-32B-Instruct-Q4_K_M.gguf"
model_path = os.path.join(DRIVE_DIR, MODEL_FILENAME)

if os.path.exists(model_path) and os.path.getsize(model_path) > 1024*1024*100:
    print(f"      🎯 Reusing permanently cached model in 5 TB Drive: {MODEL_FILENAME}")
    print(f"         Size: {os.path.getsize(model_path)/(1024**3):.2f} GB")
else:
    print(f"      📥 Streaming {MODEL_FILENAME} directly to your 5 TB Google Drive...")
    print("         (Using 16 parallel threads at 150+ MB/s, takes ~2 minutes)")
    !wget -c --show-progress -O "{model_path}" "{MODEL_URL}"
    print(f"      ✅ Saved permanently to: {model_path}")

# 4. Launch OpenAI Server & Cloudflare Public HTTPS Tunnel
print("\n[4/4] 🌐 Starting OpenAI-Compatible Server & Cloudflare Tunnel...")
API_KEY = "sk-cinemabox70b"

server_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", model_path,
    "--n_gpu_layers", "35",
    "--n_ctx", "8192",
    "--host", "127.0.0.1",
    "--port", "8000",
    "--api_key", API_KEY,
    "--chat_format", "chatml"
]

server_proc = subprocess.Popen(server_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10) # Initialize CUDA GPU weights

# Start Cloudflare tunnel
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cf_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*70)
    print("🎉 YOUR AI SERVER IS LIVE GLOBALLY AND READY TO USE!")
    print("="*70)
    print(f"\n👉 📱 CinemaBox App Endpoint:     {public_url}/v1/chat/completions")
    print(f"👉 💻 VS Code (Roo/Continue) Base: {public_url}/v1")
    print(f"👉 🔑 API Key:                     {API_KEY}")
    print(f"👉 🧠 Active Model:                Qwen2.5-32B-Instruct")
    print("\n" + "-"*70)
    print("📱 HOW TO USE IN CINEMABOX APP:")
    print(f"   1. Open CinemaBox on your phone > Settings > On-Device Intelligence")
    print(f"   2. Tap the '141B MoE / VISION' tab")
    print(f"   3. Paste '{public_url}/v1/chat/completions' into Host API Endpoint")
    print(f"   4. Tap Save & Run Test -> You are connected!")
    print("\n💻 HOW TO USE IN VS CODE (Roo Code / Continue / Cline):")
    print(f"   Provider: OpenAI Compatible")
    print(f"   Base URL: {public_url}/v1")
    print(f"   API Key:  {API_KEY}")
    print(f"   Model:    Qwen2.5-32B-Instruct")
    print("="*70 + "\n")
else:
    print("❌ Cloudflare tunnel could not be resolved. Please re-run the cell.")